# 07. 証明の生成と検証

**前提ファイル**: `witness.json`, `network.compiled`, `test.pk`, `test.vk`, `settings.json`
（`05_witness.ipynb`, `06_setup.ipynb` まで実行済み）

In [ ]:
try:
    import google.colab, subprocess, sys
    subprocess.check_call([sys.executable, "-m", "pip", "install", "ezkl", "onnx", "torch", "torchvision"])
except ImportError:
    pass

In [ ]:
import ezkl, json, os

compiled_model_path = 'network.compiled'
witness_path        = 'witness.json'
pk_path             = 'test.pk'
vk_path             = 'test.vk'
settings_path       = 'settings.json'
proof_path          = 'proof.json'

for f in [compiled_model_path, witness_path, pk_path, vk_path, settings_path]:
    assert os.path.exists(f), f"{f} がありません。"

## `prove` — 証明の生成

`prove` は以下を受け取って証明書（`proof.json`）を生成します：

```
witness.json      → 実際の計算値（中間値含む）
network.compiled  → ZK 回路（制約の集まり）
test.pk           → 証明鍵
        ↓
  proof.json  （Groth16 証明 = 楕円曲線上の点 A, B, C）
```

RareSkills の「witness `a` と制約から Groth16 証明 `(A, B, C)` を生成する」ステップです。

In [ ]:
print("証明を生成中...（数分かかる場合があります）")
res = ezkl.prove(
    witness=witness_path,
    model=compiled_model_path,
    pk_path=pk_path,
    proof_path=proof_path,
)
assert os.path.exists(proof_path)
print(f"証明生成完了: {os.path.getsize(proof_path):,} bytes")

## `proof.json` の中身を確認する

In [ ]:
with open(proof_path) as f:
    proof = json.load(f)

print(f"キー: {list(proof.keys())}\n")

if 'proof' in proof:
    print(f"proof（Groth16 の楕円曲線点、hex）: {len(proof['proof'])} 文字")

if 'instances' in proof:
    print(f"instances（公開値）: {len(proof['instances'])} 個")
    for i, inst in enumerate(proof['instances']):
        print(f"  instances[{i}]: {len(inst)} 個の値")

## `verify` — 検証

検証者は **VK と proof.json だけ**で検証できます。

- `witness.json`（中間計算値）は**不要**
- モデルの詳細も**不要**

これが ZK の本質：**秘密を見せずに「正しい」を証明できる**

In [ ]:
res = ezkl.verify(
    proof_path=proof_path,
    settings_path=settings_path,
    vk_path=vk_path
)
print(f"検証結果: {'OK — 証明は正しい' if res else 'FAIL — 証明が無効'}")

## 全ステップの対応表

```
simple_demo の処理          学習ノートブック
─────────────────────────────────────────────
モデル訓練・ONNX出力    →  01_onnx.ipynb
gen_settings            →  02_settings.ipynb
calibrate_settings      →  03_calibrate.ipynb
compile_circuit         →  04_compile.ipynb
gen_witness             →  05_witness.ipynb
get_srs + setup         →  06_setup.ipynb
prove + verify          →  07_prove_verify.ipynb
```

RareSkills との対応：
```
R1CS 制約への変換    → compile_circuit
Witness ベクトル     → gen_witness
Trusted Setup       → get_srs + setup
Groth16 証明生成    → prove
検証                → verify
```

## 次の実験アイデア
- `param_visibility = 'private'` に変える → モデル重みを隠した証明
- 森林・カーボンクレジット文脈の小さなモデルに置き換える
- `ezkl.export_solidity` で Solidity 検証コントラクトを生成してみる